In [65]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

In [66]:
df = pd.read_csv('/content/Pakwheels_feature_done.csv')

In [67]:
df.head(3)

,year,Km_travelled,fuel_type,city,assembly,engine,body_type,interior_scores,exterior_scores,comfort_scores,company,price
0,2020,50000,5.0,2.0,0.0,2000,2.0,0.0,0.0,0.0,23.0,0.64
1,2013,90000,5.0,0.0,0.0,1800,8.0,2.0,0.0,0.0,28.0,0.97
2,2011,94000,5.0,4.0,0.0,2100,8.0,2.0,2.0,0.0,28.0,0.92


In [68]:
X = df.drop(columns='price')
y = np.log1p(df['price'])

In [69]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [70]:
transformer = ColumnTransformer(transformers=[
    ('one_hot',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['fuel_type','city','assembly','body_type','company']),
    ('scale',StandardScaler(),['Km_travelled','engine','year'])
],remainder='passthrough')

In [71]:
# pipeline = Pipeline(steps=[
#     ('preprocessor',transformer),
#     ('model',RandomForestRegressor(n_estimators=100,random_state=42))
# ])

In [72]:
pipeline = Pipeline(steps=[
    ('preprocessor',transformer),
    ('model',SVR())
])

In [73]:
pipeline.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('one_hot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['fuel_type', 'city',
                                                   'assembly', 'body_type',
                                                   'company']),
                                                 ('scale', StandardScaler(),
                                                  ['Km_travelled', 'engine',
                                                   'year'])])),
                ('model', SVR())])

In [74]:
r2_score(y_test,pipeline.predict(X_test))

0.9549406089035236

In [75]:
# k_fold = KFold(n_splits=10,shuffle=True,random_state=42)
# cross_val_score(pipeline,X,y,cv=k_fold,scoring='r2').mean()

In [76]:
mean_absolute_error(np.expm1(y_test),np.expm1(pipeline.predict(X_test)))

0.07591385583718799